# Instrumented Gemma development run — Colab native execution (Protocol v2)

**Authorized Environment Specification (Protocol v2)**:
- `torch==2.14.1+cu130`
- `torchvision==0.29.1+cu130`
- `transformers==5.22.1`
- `tokenizers==0.23.3`
- `bitsandbytes==0.50.2`
- `accelerate==1.15.0`
- `huggingface_hub==1.33.0`

Run cells in order in a **fresh Python >=3.11 Colab T4 GPU session**.
Task: `marshmallow-code__marshmallow-1359` | Condition: `A` | Exactly one generation.
All setup errors are fatal; there is no template, precision, model, or CPU fallback.


In [ ]:
import sys, subprocess, importlib.metadata
if sys.version_info < (3, 11):
    raise RuntimeError("Transformers 5.22.1 requires Python >=3.11; select a compatible fresh runtime.")

PINS = [
    "torch==2.14.1+cu130",
    "torchvision==0.29.1+cu130",
    "transformers==5.22.1",
    "tokenizers==0.23.3",
    "bitsandbytes==0.50.2",
    "accelerate==1.15.0",
    "huggingface_hub==1.33.0"
]
print("Protocol v2 Pinned Dependencies:")
for p in PINS:
    print(f"  {p}")
print("\nVerifying imports in clean kernel...")


In [ ]:
import os, json, hashlib, time, uuid, platform, traceback, zipfile, contextlib, importlib.metadata, subprocess
from pathlib import Path
import torch
import transformers
import bitsandbytes as bnb
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig, GenerationConfig
from huggingface_hub import HfApi, snapshot_download

WORKSPACE_BASE = Path("/content") if Path("/content").is_dir() else Path("/kaggle/working")
if not WORKSPACE_BASE.is_dir():
    raise RuntimeError("Workspace base directory not found.")
RUN_ID = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()) + "_" + uuid.uuid4().hex[:8]
OUT = WORKSPACE_BASE / ("gemma_development_" + RUN_ID)
OUT.mkdir(exist_ok=False)
REPO = WORKSPACE_BASE / ("gemma_base_" + RUN_ID)
MODEL_ID = "google/gemma-4-E4B-it"
REVISION = "4ecc2ba63bcac23ba4a767fc87025b1450f84c9c"
EXPECTED = {
    "torch": "2.14.1",
    "torchvision": "0.29.1",
    "transformers": "5.22.1",
    "tokenizers": "0.23.3",
    "bitsandbytes": "0.50.2",
    "accelerate": "1.15.0",
    "huggingface_hub": "1.33.0"
}

def save_json(name, value):
    (OUT / name).write_text(json.dumps(value, indent=2, default=str), encoding="utf-8")

def sha_bytes(data):
    return hashlib.sha256(data).hexdigest()

def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def validate_frozen_source(packet, repo):
    blocks, source_report = [], []
    for span in packet["spans"]:
        relative = Path(span["path"])
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("Invalid evidence source path.")
        source_path = repo / relative
        if not source_path.resolve().is_relative_to(repo.resolve()):
            raise RuntimeError("Evidence symlink escapes the base repository.")
        content = "".join(source_path.read_text().splitlines(keepends=True)[span["start"]-1:span["end"]])
        if sha_bytes(content.encode()) != span["source_sha256"]:
            raise RuntimeError(f"Source mismatch: {span}")
        blocks.append(f"### Entity: {span['name']} ({span['path']}:{span['start']}-{span['end']})\n```python\n{content}\n```\n")
        source_report.append(dict(span, verified=True))
    if not blocks or "\n".join(blocks) != packet["evidence_text"] or sha_bytes(packet["evidence_text"].encode()) != packet["evidence_sha256"]:
        raise RuntimeError("Evidence is not exactly the declared source slices and wrappers.")
    return source_report

def export_bundle():
    import shutil
    files = [p for p in sorted(OUT.rglob("*")) if p.is_file() and p.name != "artifact_manifest.json"]
    save_json("artifact_manifest.json", [{"path":str(p.relative_to(OUT)), "sha256":file_sha(p), "bytes":p.stat().st_size} for p in files])
    target = OUT.with_suffix(".zip")
    with zipfile.ZipFile(target, "w", zipfile.ZIP_DEFLATED) as archive:
        for p in sorted(OUT.rglob("*")):
            if p.is_file():
                archive.write(p, p.relative_to(OUT))
    print("Artifacts:", target)
    drive_dir = Path("/content/drive/MyDrive/gemma_paper")
    if drive_dir.is_dir():
        drive_target = drive_dir / target.name
        shutil.copy2(target, drive_target)
        print("Mirrored to Drive:", drive_target)
    return target

@contextlib.contextmanager
def step(name):
    save_json("run_state.json", {"phase":name, "status":"STARTED", "run_id":RUN_ID})
    try:
        yield
    except Exception as error:
        message, trace = str(error), traceback.format_exc()
        for secret in (globals().get("TOKEN"), os.environ.get("HF_TOKEN")):
            if secret:
                message, trace = message.replace(secret, "[REDACTED]"), trace.replace(secret, "[REDACTED]")
        save_json("run_state.json", {"phase":name, "status":"FAILED", "type":type(error).__name__, "error":message})
        (OUT / "failure_traceback.txt").write_text(trace)
        export_bundle()
        raise
    save_json("run_state.json", {"phase":name, "status":"PASSED", "run_id":RUN_ID})

with step("runtime"):
    versions = {d.metadata["Name"]:d.version for d in importlib.metadata.distributions()}
    save_json("package_versions.json", versions)  # Names/versions only; no environment secrets or credential URLs.
    protocol_manifest = {
        "protocol_version": "v2_colab_native",
        "base_protocol": "kaggle_fresh_development_instrumented_v1",
        "authorized_environment_change": True,
        "task_id": "marshmallow-code__marshmallow-1359",
        "condition": "A",
        "installed_pins": {
            "torch": "2.14.1+cu130",
            "torchvision": "0.29.1+cu130",
            "transformers": "5.22.1",
            "tokenizers": "0.23.3",
            "bitsandbytes": "0.50.2",
            "accelerate": "1.15.0",
            "huggingface_hub": "1.33.0"
        },
        "version_differences": [
            "torch: 2.11.0 -> 2.14.1+cu130 (CUDA 13.0 native build)",
            "torchvision: 0.26.0 -> 0.29.1+cu130",
            "transformers: 5.22.0 -> 5.22.1",
            "tokenizers: 0.23.1 -> 0.23.3"
        ]
    }
    protocol_manifest["config_hash"] = hashlib.sha256(json.dumps(protocol_manifest, sort_keys=True).encode()).hexdigest()
    save_json("protocol_v2_manifest.json", protocol_manifest)
    for name, version in EXPECTED.items():
        actual = importlib.metadata.version(name)
        if actual.split("+")[0] != version:
            raise RuntimeError(f"Version mismatch: {name}={actual}; expected {version}")
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is unavailable. No CPU or alternative precision fallback is permitted.")
    save_json("runtime.json", {"python":platform.python_version(), "platform":platform.platform(), "torch":torch.__version__, "cuda":torch.version.cuda,
        "gpus":[{"index":i,"name":torch.cuda.get_device_name(i),"capability":torch.cuda.get_device_capability(i),"total_bytes":torch.cuda.get_device_properties(i).total_memory} for i in range(torch.cuda.device_count())]})
print("Run directory:", OUT)

## Frozen task and evidence

This sanitized metadata contains only task identifiers, base commit, issue statement, and source evidence. It does not load the benchmark dataset or gold fields in Kaggle. The selection manifest lists eligible IDs so the declared rule is independently inspectable. The packet's overlap, if any, is retained from the unchanged ranker/packer; retrieval is not tuned using the pilot's gold answers.

In [ ]:
PACKET = json.loads('{"selection": {"selection_rule": "Lexicographically smallest instance_id in pinned SWE-bench_Lite dev after excluding exactly the three pilot IDs; only instance_id/repo/base_commit/problem_statement used.", "dataset_revision": "6ec7bb89b9342f664a54a6e0a6ea6501d3437cc2", "excluded": ["marshmallow-code__marshmallow-1343", "pvlib__pvlib-python-1072", "pydicom__pydicom-1139"], "eligible_ids": ["marshmallow-code__marshmallow-1359", "pvlib__pvlib-python-1154", "pvlib__pvlib-python-1606", "pvlib__pvlib-python-1707", "pvlib__pvlib-python-1854", "pydicom__pydicom-1256", "pydicom__pydicom-1413", "pydicom__pydicom-1694", "pydicom__pydicom-901", "pylint-dev__astroid-1196", "pylint-dev__astroid-1268", "pylint-dev__astroid-1333", "pylint-dev__astroid-1866", "pylint-dev__astroid-1978", "pyvista__pyvista-4315", "sqlfluff__sqlfluff-1517", "sqlfluff__sqlfluff-1625", "sqlfluff__sqlfluff-1733", "sqlfluff__sqlfluff-1763", "sqlfluff__sqlfluff-2419"], "task": {"instance_id": "marshmallow-code__marshmallow-1359", "repo": "marshmallow-code/marshmallow", "base_commit": "b40a0f4e33823e6d0f341f7e8684e359a99060d1", "problem_statement": "3.0: DateTime fields cannot be used as inner field for List or Tuple fields\\nBetween releases 3.0.0rc8 and 3.0.0rc9, `DateTime` fields have started throwing an error when being instantiated as inner fields of container fields like `List` or `Tuple`. The snippet below works in <=3.0.0rc8 and throws the error below in >=3.0.0rc9 (and, worryingly, 3.0.0):\\r\\n\\r\\n```python\\r\\nfrom marshmallow import fields, Schema\\r\\n\\r\\nclass MySchema(Schema):\\r\\n    times = fields.List(fields.DateTime())\\r\\n\\r\\ns = MySchema()\\r\\n```\\r\\n\\r\\nTraceback:\\r\\n```\\r\\nTraceback (most recent call last):\\r\\n  File \\"test-mm.py\\", line 8, in <module>\\r\\n    s = MySchema()\\r\\n  File \\"/Users/victor/.pyenv/versions/marshmallow/lib/python3.6/site-packages/marshmallow/schema.py\\", line 383, in __init__\\r\\n    self.fields = self._init_fields()\\r\\n  File \\"/Users/victor/.pyenv/versions/marshmallow/lib/python3.6/site-packages/marshmallow/schema.py\\", line 913, in _init_fields\\r\\n    self._bind_field(field_name, field_obj)\\r\\n  File \\"/Users/victor/.pyenv/versions/marshmallow/lib/python3.6/site-packages/marshmallow/schema.py\\", line 969, in _bind_field\\r\\n    field_obj._bind_to_schema(field_name, self)\\r\\n  File \\"/Users/victor/.pyenv/versions/marshmallow/lib/python3.6/site-packages/marshmallow/fields.py\\", line 636, in _bind_to_schema\\r\\n    self.inner._bind_to_schema(field_name, self)\\r\\n  File \\"/Users/victor/.pyenv/versions/marshmallow/lib/python3.6/site-packages/marshmallow/fields.py\\", line 1117, in _bind_to_schema\\r\\n    or getattr(schema.opts, self.SCHEMA_OPTS_VAR_NAME)\\r\\nAttributeError: \'List\' object has no attribute \'opts\'\\r\\n```\\r\\n\\r\\nIt seems like it\'s treating the parent field as a Schema without checking that it is indeed a schema, so the `schema.opts` statement fails as fields don\'t have an `opts` attribute.\\n"}}, "condition": "A", "condition_rule": "Original complete function/class packing; retrieval unchanged and packet frozen before GPU run", "evidence_text": "### Entity: SchemaMeta (src/marshmallow/schema.py:84-184)\\n```python\\nclass SchemaMeta(type):\\n    \\"\\"\\"Metaclass for the Schema class. Binds the declared fields to\\n    a ``_declared_fields`` attribute, which is a dictionary mapping attribute\\n    names to field objects. Also sets the ``opts`` class attribute, which is\\n    the Schema class\'s ``class Meta`` options.\\n    \\"\\"\\"\\n\\n    def __new__(mcs, name, bases, attrs):\\n        meta = attrs.get(\\"Meta\\")\\n        ordered = getattr(meta, \\"ordered\\", False)\\n        if not ordered:\\n            # Inherit \'ordered\' option\\n            # Warning: We loop through bases instead of MRO because we don\'t\\n            # yet have access to the class object\\n            # (i.e. can\'t call super before we have fields)\\n            for base_ in bases:\\n                if hasattr(base_, \\"Meta\\") and hasattr(base_.Meta, \\"ordered\\"):\\n                    ordered = base_.Meta.ordered\\n                    break\\n            else:\\n                ordered = False\\n        cls_fields = _get_fields(attrs, base.FieldABC, pop=True, ordered=ordered)\\n        klass = super().__new__(mcs, name, bases, attrs)\\n        inherited_fields = _get_fields_by_mro(klass, base.FieldABC, ordered=ordered)\\n\\n        meta = klass.Meta\\n        # Set klass.opts in __new__ rather than __init__ so that it is accessible in\\n        # get_declared_fields\\n        klass.opts = klass.OPTIONS_CLASS(meta, ordered=ordered)\\n        # Add fields specified in the `include` class Meta option\\n        cls_fields += list(klass.opts.include.items())\\n\\n        dict_cls = OrderedDict if ordered else dict\\n        # Assign _declared_fields on class\\n        klass._declared_fields = mcs.get_declared_fields(\\n            klass=klass,\\n            cls_fields=cls_fields,\\n            inherited_fields=inherited_fields,\\n            dict_cls=dict_cls,\\n        )\\n        return klass\\n\\n    @classmethod\\n    def get_declared_fields(mcs, klass, cls_fields, inherited_fields, dict_cls):\\n        \\"\\"\\"Returns a dictionary of field_name => `Field` pairs declard on the class.\\n        This is exposed mainly so that plugins can add additional fields, e.g. fields\\n        computed from class Meta options.\\n\\n        :param type klass: The class object.\\n        :param list cls_fields: The fields declared on the class, including those added\\n            by the ``include`` class Meta option.\\n        :param list inherited_fields: Inherited fields.\\n        :param type dict_class: Either `dict` or `OrderedDict`, depending on the whether\\n            the user specified `ordered=True`.\\n        \\"\\"\\"\\n        return dict_cls(inherited_fields + cls_fields)\\n\\n    def __init__(cls, name, bases, attrs):\\n        super().__init__(cls, bases, attrs)\\n        if name and cls.opts.register:\\n            class_registry.register(name, cls)\\n        cls._hooks = cls.resolve_hooks()\\n\\n    def resolve_hooks(cls):\\n        \\"\\"\\"Add in the decorated processors\\n\\n        By doing this after constructing the class, we let standard inheritance\\n        do all the hard work.\\n        \\"\\"\\"\\n        mro = inspect.getmro(cls)\\n\\n        hooks = defaultdict(list)\\n\\n        for attr_name in dir(cls):\\n            # Need to look up the actual descriptor, not whatever might be\\n            # bound to the class. This needs to come from the __dict__ of the\\n            # declaring class.\\n            for parent in mro:\\n                try:\\n                    attr = parent.__dict__[attr_name]\\n                except KeyError:\\n                    continue\\n                else:\\n                    break\\n            else:\\n                # In case we didn\'t find the attribute and didn\'t break above.\\n                # We should never hit this - it\'s just here for completeness\\n                # to exclude the possibility of attr being undefined.\\n                continue\\n\\n            try:\\n                hook_config = attr.__marshmallow_hook__\\n            except AttributeError:\\n                pass\\n            else:\\n                for key in hook_config.keys():\\n                    # Use name here so we can get the bound method later, in\\n                    # case the processor was a descriptor or something.\\n                    hooks[key].append(attr_name)\\n\\n        return hooks\\n\\n```\\n\\n### Entity: List (src/marshmallow/fields.py:598-667)\\n```python\\nclass List(Field):\\n    \\"\\"\\"A list field, composed with another `Field` class or\\n    instance.\\n\\n    Example: ::\\n\\n        numbers = fields.List(fields.Float())\\n\\n    :param Field cls_or_instance: A field class or instance.\\n    :param bool default: Default value for serialization.\\n    :param kwargs: The same keyword arguments that :class:`Field` receives.\\n\\n    .. versionchanged:: 2.0.0\\n        The ``allow_none`` parameter now applies to deserialization and\\n        has the same semantics as the other fields.\\n\\n    .. versionchanged:: 3.0.0rc9\\n        Does not serialize scalar values to single-item lists.\\n    \\"\\"\\"\\n\\n    default_error_messages = {\\"invalid\\": \\"Not a valid list.\\"}\\n\\n    def __init__(self, cls_or_instance, **kwargs):\\n        super().__init__(**kwargs)\\n        try:\\n            self.inner = resolve_field_instance(cls_or_instance)\\n        except FieldInstanceResolutionError as error:\\n            raise ValueError(\\n                \\"The list elements must be a subclass or instance of \\"\\n                \\"marshmallow.base.FieldABC.\\"\\n            ) from error\\n        if isinstance(self.inner, Nested):\\n            self.only = self.inner.only\\n            self.exclude = self.inner.exclude\\n\\n    def _bind_to_schema(self, field_name, schema):\\n        super()._bind_to_schema(field_name, schema)\\n        self.inner = copy.deepcopy(self.inner)\\n        self.inner._bind_to_schema(field_name, self)\\n        if isinstance(self.inner, Nested):\\n            self.inner.only = self.only\\n            self.inner.exclude = self.exclude\\n\\n    def _serialize(self, value, attr, obj, **kwargs):\\n        if value is None:\\n            return None\\n        # Optimize dumping a list of Nested objects by calling dump(many=True)\\n        if isinstance(self.inner, Nested) and not self.inner.many:\\n            return self.inner._serialize(value, attr, obj, many=True, **kwargs)\\n        return [self.inner._serialize(each, attr, obj, **kwargs) for each in value]\\n\\n    def _deserialize(self, value, attr, data, **kwargs):\\n        if not utils.is_collection(value):\\n            raise self.make_error(\\"invalid\\")\\n        # Optimize loading a list of Nested objects by calling load(many=True)\\n        if isinstance(self.inner, Nested) and not self.inner.many:\\n            return self.inner.deserialize(value, many=True, **kwargs)\\n\\n        result = []\\n        errors = {}\\n        for idx, each in enumerate(value):\\n            try:\\n                result.append(self.inner.deserialize(each, **kwargs))\\n            except ValidationError as error:\\n                if error.valid_data is not None:\\n                    result.append(error.valid_data)\\n                errors.update({idx: error.messages})\\n        if errors:\\n            raise ValidationError(errors, valid_data=result)\\n        return result\\n\\n```\\n\\n### Entity: Meta (src/marshmallow/schema.py:311-351)\\n```python\\n    class Meta:\\n        \\"\\"\\"Options object for a Schema.\\n\\n        Example usage: ::\\n\\n            class Meta:\\n                fields = (\\"id\\", \\"email\\", \\"date_created\\")\\n                exclude = (\\"password\\", \\"secret_attribute\\")\\n\\n        Available options:\\n\\n        - ``fields``: Tuple or list of fields to include in the serialized result.\\n        - ``additional``: Tuple or list of fields to include *in addition* to the\\n            explicitly declared fields. ``additional`` and ``fields`` are\\n            mutually-exclusive options.\\n        - ``include``: Dictionary of additional fields to include in the schema. It is\\n            usually better to define fields as class variables, but you may need to\\n            use this option, e.g., if your fields are Python keywords. May be an\\n            `OrderedDict`.\\n        - ``exclude``: Tuple or list of fields to exclude in the serialized result.\\n            Nested fields can be represented with dot delimiters.\\n        - ``dateformat``: Default format for `Date <fields.Date>` fields.\\n        - ``datetimeformat``: Default format for `DateTime <fields.DateTime>` fields.\\n        - ``render_module``: Module to use for `loads <Schema.loads>` and `dumps <Schema.dumps>`.\\n            Defaults to `json` from the standard library.\\n        - ``ordered``: If `True`, order serialization output according to the\\n            order in which fields were declared. Output of `Schema.dump` will be a\\n            `collections.OrderedDict`.\\n        - ``index_errors``: If `True`, errors dictionaries will include the index\\n            of invalid items in a collection.\\n        - ``load_only``: Tuple or list of fields to exclude from serialized results.\\n        - ``dump_only``: Tuple or list of fields to exclude from deserialization\\n        - ``unknown``: Whether to exclude, include, or raise an error for unknown\\n            fields in the data. Use `EXCLUDE`, `INCLUDE` or `RAISE`.\\n        - ``register``: Whether to register the `Schema` with marshmallow\'s internal\\n            class registry. Must be `True` if you intend to refer to this `Schema`\\n            by class name in `Nested` fields. Only set this to `False` when memory\\n            usage is critical. Defaults to `True`.\\n        \\"\\"\\"\\n\\n        pass\\n\\n```\\n\\n### Entity: DateTime (src/marshmallow/fields.py:1067-1153)\\n```python\\nclass DateTime(Field):\\n    \\"\\"\\"A formatted datetime string.\\n\\n    Example: ``\'2014-12-22T03:12:58.019077+00:00\'``\\n\\n    :param str format: Either ``\\"rfc\\"`` (for RFC822), ``\\"iso\\"`` (for ISO8601),\\n        or a date format string. If `None`, defaults to \\"iso\\".\\n    :param kwargs: The same keyword arguments that :class:`Field` receives.\\n\\n    .. versionchanged:: 3.0.0rc9\\n        Does not modify timezone information on (de)serialization.\\n    \\"\\"\\"\\n\\n    SERIALIZATION_FUNCS = {\\n        \\"iso\\": utils.isoformat,\\n        \\"iso8601\\": utils.isoformat,\\n        \\"rfc\\": utils.rfcformat,\\n        \\"rfc822\\": utils.rfcformat,\\n    }\\n\\n    DESERIALIZATION_FUNCS = {\\n        \\"iso\\": utils.from_iso_datetime,\\n        \\"iso8601\\": utils.from_iso_datetime,\\n        \\"rfc\\": utils.from_rfc,\\n        \\"rfc822\\": utils.from_rfc,\\n    }\\n\\n    DEFAULT_FORMAT = \\"iso\\"\\n\\n    OBJ_TYPE = \\"datetime\\"\\n\\n    SCHEMA_OPTS_VAR_NAME = \\"datetimeformat\\"\\n\\n    default_error_messages = {\\n        \\"invalid\\": \\"Not a valid {obj_type}.\\",\\n        \\"invalid_awareness\\": \\"Not a valid {awareness} {obj_type}.\\",\\n        \\"format\\": \'\\"{input}\\" cannot be formatted as a {obj_type}.\',\\n    }\\n\\n    def __init__(self, format=None, **kwargs):\\n        super().__init__(**kwargs)\\n        # Allow this to be None. It may be set later in the ``_serialize``\\n        # or ``_deserialize`` methods. This allows a Schema to dynamically set the\\n        # format, e.g. from a Meta option\\n        self.format = format\\n\\n    def _bind_to_schema(self, field_name, schema):\\n        super()._bind_to_schema(field_name, schema)\\n        self.format = (\\n            self.format\\n            or getattr(schema.opts, self.SCHEMA_OPTS_VAR_NAME)\\n            or self.DEFAULT_FORMAT\\n        )\\n\\n    def _serialize(self, value, attr, obj, **kwargs):\\n        if value is None:\\n            return None\\n        data_format = self.format or self.DEFAULT_FORMAT\\n        format_func = self.SERIALIZATION_FUNCS.get(data_format)\\n        if format_func:\\n            return format_func(value)\\n        else:\\n            return value.strftime(data_format)\\n\\n    def _deserialize(self, value, attr, data, **kwargs):\\n        if not value:  # Falsy values, e.g. \'\', None, [] are not valid\\n            raise self.make_error(\\"invalid\\", input=value, obj_type=self.OBJ_TYPE)\\n        data_format = self.format or self.DEFAULT_FORMAT\\n        func = self.DESERIALIZATION_FUNCS.get(data_format)\\n        if func:\\n            try:\\n                return func(value)\\n            except (TypeError, AttributeError, ValueError) as error:\\n                raise self.make_error(\\n                    \\"invalid\\", input=value, obj_type=self.OBJ_TYPE\\n                ) from error\\n        else:\\n            try:\\n                return self._make_object_from_format(value, data_format)\\n            except (TypeError, AttributeError, ValueError) as error:\\n                raise self.make_error(\\n                    \\"invalid\\", input=value, obj_type=self.OBJ_TYPE\\n                ) from error\\n\\n    @staticmethod\\n    def _make_object_from_format(value, data_format):\\n        return dt.datetime.strptime(value, data_format)\\n\\n```\\n\\n### Entity: _bind_field (src/marshmallow/schema.py:957-979)\\n```python\\n    def _bind_field(self, field_name, field_obj):\\n        \\"\\"\\"Bind field to the schema, setting any necessary attributes on the\\n        field (e.g. parent and name).\\n\\n        Also set field load_only and dump_only values if field_name was\\n        specified in ``class Meta``.\\n        \\"\\"\\"\\n        try:\\n            if field_name in self.load_only:\\n                field_obj.load_only = True\\n            if field_name in self.dump_only:\\n                field_obj.dump_only = True\\n            field_obj._bind_to_schema(field_name, self)\\n            self.on_bind_field(field_name, field_obj)\\n        except TypeError as error:\\n            # field declared as a class, not an instance\\n            if isinstance(field_obj, type) and issubclass(field_obj, base.FieldABC):\\n                msg = (\\n                    \'Field for \\"{}\\" must be declared as a \'\\n                    \\"Field instance, not a class. \\"\\n                    \'Did you mean \\"fields.{}()\\"?\'.format(field_name, field_obj.__name__)\\n                )\\n                raise TypeError(msg) from error\\n\\n```\\n\\n### Entity: __new__ (src/marshmallow/schema.py:91-124)\\n```python\\n    def __new__(mcs, name, bases, attrs):\\n        meta = attrs.get(\\"Meta\\")\\n        ordered = getattr(meta, \\"ordered\\", False)\\n        if not ordered:\\n            # Inherit \'ordered\' option\\n            # Warning: We loop through bases instead of MRO because we don\'t\\n            # yet have access to the class object\\n            # (i.e. can\'t call super before we have fields)\\n            for base_ in bases:\\n                if hasattr(base_, \\"Meta\\") and hasattr(base_.Meta, \\"ordered\\"):\\n                    ordered = base_.Meta.ordered\\n                    break\\n            else:\\n                ordered = False\\n        cls_fields = _get_fields(attrs, base.FieldABC, pop=True, ordered=ordered)\\n        klass = super().__new__(mcs, name, bases, attrs)\\n        inherited_fields = _get_fields_by_mro(klass, base.FieldABC, ordered=ordered)\\n\\n        meta = klass.Meta\\n        # Set klass.opts in __new__ rather than __init__ so that it is accessible in\\n        # get_declared_fields\\n        klass.opts = klass.OPTIONS_CLASS(meta, ordered=ordered)\\n        # Add fields specified in the `include` class Meta option\\n        cls_fields += list(klass.opts.include.items())\\n\\n        dict_cls = OrderedDict if ordered else dict\\n        # Assign _declared_fields on class\\n        klass._declared_fields = mcs.get_declared_fields(\\n            klass=klass,\\n            cls_fields=cls_fields,\\n            inherited_fields=inherited_fields,\\n            dict_cls=dict_cls,\\n        )\\n        return klass\\n\\n```\\n\\n### Entity: Raw (src/marshmallow/fields.py:402-405)\\n```python\\nclass Raw(Field):\\n    \\"\\"\\"Field that applies no formatting or validation.\\"\\"\\"\\n\\n    pass\\n\\n```\\n", "evidence_tokens": 3998, "packer_reported_tokens": 3998, "evidence_sha256": "739b3ee069de9b9d484423fe11c4d62f1ffd9c749483b2eee1fd45a8de65f3d7", "spans": [{"name": "SchemaMeta", "path": "src/marshmallow/schema.py", "start": 84, "end": 184, "source_sha256": "1ea7ded7c9cd0ee87364492e7eabcb24afce583c85616ba7ed3d99517eac71ea"}, {"name": "List", "path": "src/marshmallow/fields.py", "start": 598, "end": 667, "source_sha256": "9ee64cdcce6294228adb9801b41eddf6f2826d9d30292dbb885ba20107e4cfc1"}, {"name": "Meta", "path": "src/marshmallow/schema.py", "start": 311, "end": 351, "source_sha256": "4fe6f9de66a569ea81326f5464b84e9d406bc687f9e2421d104c7206a1bd88c1"}, {"name": "DateTime", "path": "src/marshmallow/fields.py", "start": 1067, "end": 1153, "source_sha256": "429f0d294b55e4180f1e809e5fb7c7e8e70b30c9364a4985e786c8d79ee4cfc8"}, {"name": "_bind_field", "path": "src/marshmallow/schema.py", "start": 957, "end": 979, "source_sha256": "bf9d6fb6bbd5c3b34e262b349cd8496513927ff0854adcfd7f9e8ffaa0001480"}, {"name": "__new__", "path": "src/marshmallow/schema.py", "start": 91, "end": 124, "source_sha256": "f1e9243a05e87159553d429e669943cef6333ab9c1b00b841b7472518a8f0673"}, {"name": "Raw", "path": "src/marshmallow/fields.py", "start": 402, "end": 405, "source_sha256": "47e40a08c05f32aa281ca11edb6e4de0ff3192a032c3efae7d4600897870208a"}], "retrieval_source_sha256": "f7c94f395474583054aad4c367cd65a1bdf876bc2b00e3d21c3b62a98b3b0ff8", "ranking_sha256": "eb6ab77ed58890f8897bb9f6006be68252309623d3362afc9f5f915fae1462a4", "packet_qa": {"budget_limit": 4000, "token_count": 3998, "under_ceiling": true, "entities_included": 7, "all_passed": true}, "provenance": "NEW_DEVELOPMENT_PACKET; not a reconstructed pilot packet; no gold selection or tuning"}')
with step("base_source_validation"):
    selection = PACKET["selection"]
    TASK = selection["task"]
    if TASK["instance_id"] in selection["excluded"] or TASK["instance_id"] != min(selection["eligible_ids"]):
        raise RuntimeError("Deterministic task selection failed.")
    if PACKET["condition"] != "A":
        raise RuntimeError("The declared single condition is A.")
    save_json("frozen_packet.json", PACKET)
    save_json("selection.json", selection)
    commands = []
    def git(*args):
        argv = ["git", "-C", str(REPO), *args]
        env = dict(os.environ, GIT_CONFIG_GLOBAL="/dev/null", GIT_CONFIG_NOSYSTEM="1", GIT_TERMINAL_PROMPT="0")
        result = subprocess.run(argv, env=env, capture_output=True, text=True, timeout=300)
        commands.append({"argv":argv,"exit_code":result.returncode,"stdout":result.stdout,"stderr":result.stderr})
        save_json("base_commands.json", commands)
        if result.returncode:
            raise RuntimeError(f"Git failed: {argv}: {result.stderr}")
        return result.stdout
    REPO.mkdir(exist_ok=False)
    git("init", "--quiet")
    git("fetch", "--depth", "1", "https://github.com/" + TASK["repo"] + ".git", TASK["base_commit"])
    git("checkout", "--detach", "--quiet", "FETCH_HEAD")
    if git("rev-parse", "HEAD").strip() != TASK["base_commit"] or git("status", "--porcelain").strip():
        raise RuntimeError("Repository is not a clean exact-base checkout.")
    source_report = validate_frozen_source(PACKET, REPO)
    save_json("source_validation.json", {"base_commit":TASK["base_commit"],"verified_spans":source_report,"retrieval_changed":False})
    (OUT / "evidence.txt").write_text(PACKET["evidence_text"])
    (OUT / "issue.txt").write_text(TASK["problem_statement"])
print(TASK["instance_id"], "source evidence verified")

In [ ]:
with step("processor_and_prompt_validation"):
    TOKEN = os.environ.get("HF_TOKEN")
    if not TOKEN:
        try:
            from google.colab import userdata
            TOKEN = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if not TOKEN:
        try:
            from kaggle_secrets import UserSecretsClient
            TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            TOKEN = None  # Anonymous authentication is allowed; model/precision/template fallbacks are not.
    resolved = HfApi(token=TOKEN).model_info(MODEL_ID, revision=REVISION).sha
    if resolved != REVISION:
        raise RuntimeError("Model revision did not resolve to the declared commit.")
    processor = AutoProcessor.from_pretrained(MODEL_ID, revision=REVISION, token=TOKEN, trust_remote_code=False)
    tokenizer = processor.tokenizer
    count = len(tokenizer.encode(PACKET["evidence_text"], add_special_tokens=False))
    if count != PACKET["evidence_tokens"] or not 0 < count <= 4000:
        raise RuntimeError(f"Evidence token validation failed: actual={count}, frozen={PACKET['evidence_tokens']}")
    PATCH_ONLY = ("\n### PATCH-ONLY INSTRUCTION — fresh instrumented protocol v1\n"
        "Return only one explicitly labelled diff fenced block containing a unified diff. "
        "Use exact repository source context. Do not include explanations, placeholders, ellipses, or text outside the diff block.\n")
    prompt = (f"You are an expert software engineer resolving an issue in the repository for task '{TASK['instance_id']}'.\n\n"
        f"### ISSUE PROBLEM STATEMENT\n{TASK['problem_statement'].strip()}\n\n"
        f"### RELEVANT REPOSITORY CODE (Evidence Condition: A)\n{PACKET['evidence_text'].strip()}\n\n"
        "### INSTRUCTIONS\nAnalyze the issue description and repository code above.\nWrite a patch that fixes the issue.\n"
        "Output your patch as a unified diff format inside a ```diff code block.\n"
        "Ensure file paths in the diff match the repository files (e.g., a/src/... b/src/...).\n" + PATCH_ONLY)
    messages = [{"role":"user", "content":prompt}]
    settings = {"tokenize":False,"add_generation_prompt":True,"enable_thinking":False}
    rendered = processor.apply_chat_template(messages, **settings)  # Exceptions are fatal: no raw-text fallback.
    rendered_thinking = processor.apply_chat_template(messages, **dict(settings, enable_thinking=True))
    if "<|think|>" in rendered or "<|think|>" not in rendered_thinking or rendered == rendered_thinking:
        raise RuntimeError("Cannot demonstrate effective thinking=False template behavior.")
    if not rendered.startswith("<bos><|turn>user\n") or not rendered.endswith("<turn|>\n<|turn>model\n"):
        raise RuntimeError("Unexpected canonical user/model turn boundaries.")
    template = processor.chat_template
    if not isinstance(template, str):
        raise RuntimeError("Expected a single inspectable canonical chat template.")
    processor_kwargs = {"add_special_tokens":False,"return_tensors":"pt","return_attention_mask":True,"return_mm_token_type_ids":True}
    inputs = dict(processor(text=[rendered], **processor_kwargs))
    if not {"input_ids","attention_mask","mm_token_type_ids"}.issubset(inputs):
        raise RuntimeError(f"Missing required text processor inputs: {list(inputs)}")
    if not all(isinstance(value, torch.Tensor) for value in inputs.values()):
        raise RuntimeError("Unexpected non-tensor processor input; inspect and explicitly support it rather than dropping it.")
    expected_ids = tokenizer.encode(rendered, add_special_tokens=False)
    if inputs["input_ids"][0].tolist() != expected_ids:
        raise RuntimeError("Processor input IDs differ from the exact rendered prompt tokenization.")
    bos = tokenizer.bos_token_id
    if bos is None or inputs["input_ids"][0,0].item() != bos or (inputs["input_ids"] == bos).sum().item() != 1:
        raise RuntimeError("Missing or duplicate BOS token.")
    torch.save({k:v.detach().cpu().clone() for k,v in inputs.items()}, OUT / "processor_inputs.pt")
    save_json("input_ids.json", inputs["input_ids"].tolist())
    save_json("messages.json", messages)
    (OUT / "prompt.txt").write_text(prompt)
    (OUT / "rendered_prompt.txt").write_text(rendered)
    (OUT / "rendered_with_thinking_true_control.txt").write_text(rendered_thinking)
    (OUT / "chat_template.jinja").write_text(template)
    (OUT / "decoded_input_with_special_tokens.txt").write_text(tokenizer.decode(expected_ids, skip_special_tokens=False))
    save_json("template_and_inputs.json", {"record_type":"NEW_EXECUTION_RECORD","model_id":MODEL_ID,"requested_revision":REVISION,"resolved_revision":resolved,
        "template_settings":settings,"processor_kwargs":processor_kwargs,"prompt_sha256":sha_bytes(prompt.encode()),"rendered_sha256":sha_bytes(rendered.encode()),
        "template_sha256":sha_bytes(template.encode()),"evidence_tokens":count,"input_tokens":len(expected_ids),
        "processor_keys":list(inputs),"inputs":{k:{"shape":list(v.shape),"dtype":str(v.dtype),"device":str(v.device)} for k,v in inputs.items()},
        "prompt_protocol":"NEW instrumented patch-only suffix; not recovered historical v2 bytes"})
print("Evidence tokens:", count, "| Actual input keys:", list(inputs), "| Input tokens:", len(expected_ids))

In [ ]:
def memory_all_gpus():
    report = []
    for device in range(torch.cuda.device_count()):
        torch.cuda.synchronize(device)
        free, total = torch.cuda.mem_get_info(device)
        report.append({"index":device,"name":torch.cuda.get_device_name(device),"free_bytes":free,"total_bytes":total,
            "allocated_bytes":torch.cuda.memory_allocated(device),"reserved_bytes":torch.cuda.memory_reserved(device),
            "peak_allocated_bytes":torch.cuda.max_memory_allocated(device),"peak_reserved_bytes":torch.cuda.max_memory_reserved(device)})
    return report

with step("strict_int8_model_load"):
    save_json("memory_before_model_load.json", memory_all_gpus())
    snapshot = Path(snapshot_download(MODEL_ID, revision=REVISION, token=TOKEN,
        allow_patterns=["*.json","*.safetensors","*.jinja"]))
    if snapshot.name != REVISION:
        raise RuntimeError("Downloaded model snapshot path is not the resolved revision.")
    shards = sorted(snapshot.glob("*.safetensors"))
    if not shards:
        raise RuntimeError("Pinned snapshot has no safetensors weights. No format/model fallback permitted.")
    inventory = [{"name":p.name,"bytes":p.stat().st_size,"sha256":file_sha(p)} for p in shards]
    save_json("loaded_snapshot.json", {"model_id":MODEL_ID,"requested_revision":REVISION,"resolved_revision":resolved,
        "snapshot_directory_name":snapshot.name,"weights":inventory,"config_sha256":file_sha(snapshot / "config.json"),
        "tokenizer_sha256":file_sha(snapshot / "tokenizer.json")})
    quantization = BitsAndBytesConfig(load_in_8bit=True, load_in_4bit=False, llm_int8_enable_fp32_cpu_offload=False)
    save_json("model_load_settings.json", {"model_class":"AutoModelForImageTextToText","revision":resolved,"local_files_only":True,
        "quantization_config":quantization.to_dict(),"dtype":"torch.float16","device_map":"auto","cpu_offload_allowed":False})
    model = AutoModelForImageTextToText.from_pretrained(str(snapshot), local_files_only=True, trust_remote_code=False,
        quantization_config=quantization, dtype=torch.float16, device_map="auto")
    model.eval()
    eight, four = [], []
    for name, module in model.named_modules():
        if isinstance(module, bnb.nn.Linear8bitLt):
            eight.append({"name":name,"weight_dtype":str(module.weight.dtype),"device":str(module.weight.device),"weight_class":type(module.weight).__name__})
        if isinstance(module, bnb.nn.Linear4bit):
            four.append(name)
    placements = [{"name":name,"shape":list(p.shape),"numel":p.numel(),"dtype":str(p.dtype),"device":str(p.device),"class":type(p).__name__} for name,p in model.named_parameters(remove_duplicate=False)]
    buffers = [{"name":name,"shape":list(b.shape),"dtype":str(b.dtype),"device":str(b.device)} for name,b in model.named_buffers()]
    weight_gpus = sorted({p.device.index for p in model.parameters() if p.device.type == "cuda"})
    quant_report = {"is_loaded_in_8bit":getattr(model,"is_loaded_in_8bit",False),"quantized_8bit_layers":eight,"quantized_4bit_layers":four,
        "hf_device_map":getattr(model,"hf_device_map",None),"weight_gpu_indices":weight_gpus,"parameters":placements,"buffers":buffers}
    save_json("quantization_and_placement.json", quant_report)
    save_json("loaded_model_config.json", model.config.to_dict())
    if not quant_report["is_loaded_in_8bit"] or not eight or four:
        raise RuntimeError("Actual loaded model did not pass strict INT8 inspection.")
    if any(row["weight_dtype"] != "torch.int8" or not row["device"].startswith("cuda:") for row in eight):
        raise RuntimeError("Some intended 8-bit weights are not GPU INT8 tensors.")
    if not weight_gpus or any(p.device.type != "cuda" for p in model.parameters()):
        raise RuntimeError("Model weights are on CPU/disk/meta; no offload fallback permitted.")
    save_json("memory_after_model_load.json", memory_all_gpus())
print("Verified INT8 layers:", len(eight), "| Every GPU holding weights:", weight_gpus)

In [ ]:
with step("generation_preflight"):
    input_device = model.get_input_embeddings().weight.device
    if input_device.type != "cuda":
        raise RuntimeError("Input embedding device is not CUDA.")
    generation_inputs = {key:value.to(input_device) for key,value in inputs.items()}
    if set(generation_inputs) != set(inputs):
        raise RuntimeError("Processor inputs were discarded.")
    for key in inputs:
        if not torch.equal(inputs[key].cpu(), generation_inputs[key].cpu()):
            raise RuntimeError(f"Input tensor changed during device transfer: {key}")
    torch.save({k:v.detach().cpu().clone() for k,v in generation_inputs.items()}, OUT / "actual_generation_inputs.pt")
    save_json("actual_generation_input_manifest.json", {k:{"shape":list(v.shape),"dtype":str(v.dtype),"device":str(v.device),
        "value_sha256":sha_bytes(v.detach().cpu().contiguous().view(torch.uint8).numpy().tobytes())} for k,v in generation_inputs.items()})
    effective = GenerationConfig.from_dict(model.generation_config.to_dict())
    unused = effective.update(max_new_tokens=2048, do_sample=False, num_beams=1, num_return_sequences=1, use_cache=True)
    if unused:
        raise RuntimeError(f"Unrecognized generation configuration: {unused}")
    if effective.max_new_tokens != 2048 or effective.do_sample or effective.num_beams != 1:
        raise RuntimeError("Effective generation settings violate the declared protocol.")
    limit = getattr(getattr(model.config,"text_config",model.config), "max_position_embeddings", None)
    if limit is None or generation_inputs["input_ids"].shape[1] + 2048 > limit:
        raise RuntimeError("Model context capacity is unknown or insufficient; no input truncation permitted.")
    save_json("effective_generation_config.json", effective.to_dict())
    save_json("generation_call.json", {"all_processor_keys_passed":list(generation_inputs),"discarded_keys":[],
        "generation_config":"effective_generation_config.json","return_dict_in_generate":True,"output_scores":False,
        "model_revision":resolved,"condition":"A","attempt":1})
    for device in range(torch.cuda.device_count()):
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
    save_json("memory_before_generation.json", memory_all_gpus())
    save_json("validation_passed.json", {"source":True,"budget":True,"template":True,"effective_thinking_false":True,"all_inputs_preserved":True,
        "actual_int8":True,"model_revision":resolved,"weight_gpus":weight_gpus,"run_id":RUN_ID})
    export_bundle()  # Persist all pre-generation evidence even if the later GPU call crashes.
print("Validation passed. The next cell performs exactly one fresh generation.")

## Single generation — run once manually

This cell is intentionally the only `model.generate` call. It refuses to overwrite output or retry a failed generation. A rejected processor key or an out-of-memory error is preserved as a failure, not worked around by dropping inputs, offloading, changing the model, or changing precision. No tests or reference patches are available here.

In [ ]:
with step("single_generation"):
    if not (OUT / "validation_passed.json").exists() or (OUT / "generation_started.json").exists():
        raise RuntimeError("Missing validation or generation already attempted; do not retry this run ID.")
    save_json("generation_started.json", {"run_id":RUN_ID,"started_unix":time.time()})
    start = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(**generation_inputs, generation_config=effective, return_dict_in_generate=True, output_scores=False)
    for device in range(torch.cuda.device_count()):
        torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - start
    sequences = generated.sequences.detach().cpu()
    torch.save(sequences, OUT / "output_sequences.pt")
    prefix_len = generation_inputs["input_ids"].shape[1]
    if not torch.equal(sequences[0,:prefix_len], generation_inputs["input_ids"][0].cpu()):
        raise RuntimeError("Generated sequence does not retain the exact input prefix.")
    new_ids = sequences[0,prefix_len:]
    save_json("output_token_ids.json", new_ids.tolist())
    raw_with_specials = processor.decode(new_ids, skip_special_tokens=False)
    raw_text = processor.decode(new_ids, skip_special_tokens=True)
    (OUT / "raw_output_with_special_tokens.txt").write_text(raw_with_specials)
    (OUT / "raw_output.txt").write_text(raw_text)
    eos = effective.eos_token_id
    eos_ids = set(eos if isinstance(eos,(list,tuple)) else [eos]) if eos is not None else set()
    ended_eos = bool(len(new_ids) and new_ids[-1].item() in eos_ids)
    hit_limit = len(new_ids) >= 2048
    save_json("memory_after_generation.json", memory_all_gpus())
    save_json("checkpoint.json", {"record_type":"FRESH_INSTRUMENTED_GPU_EXECUTION_PROTOCOL_V2","protocol_version":"v2_colab_native","instance_id":TASK["instance_id"],"base_commit":TASK["base_commit"],
        "condition":"A","attempt":1,"model_id":MODEL_ID,"actual_resolved_revision":resolved,"quantization":"verified_bitsandbytes_8bit",
        "prompt_sha256":sha_bytes(prompt.encode()),"rendered_prompt_sha256":sha_bytes(rendered.encode()),"evidence_sha256":PACKET["evidence_sha256"],
        "effective_generation_config_sha256":file_sha(OUT / "effective_generation_config.json"),"input_tokens":prefix_len,"generated_tokens":len(new_ids),
        "finish_reason":"token_limit" if hit_limit else ("eos" if ended_eos else "other_stop"),"ended_eos":ended_eos,"hit_token_cap":hit_limit,
        "elapsed_seconds":elapsed,"weight_gpu_indices":weight_gpus,"test_resolution":"NOT_EVALUATED","run_id":RUN_ID})
    print("Saved fresh output:", TASK["instance_id"], len(new_ids), "tokens; no tests executed.")
export_bundle()

In [ ]:
# Kaggle's Output panel also exposes this ZIP. It excludes repositories, weights, caches and credentials.
from IPython.display import FileLink, display
display(FileLink(str(OUT.with_suffix(".zip"))))